# CS 521 HW3, Problem 2: Arithmetic Extraction with SynCode

Phi-2 with a SynCode grammar extracts the two operands and the operator from a query, and Python computes the answer. Run on a Colab GPU (Runtime > Change runtime type > T4 GPU), then run all cells.

# Setup

Colab runs Python 3.13, which the SynCode release on PyPI does not support, so SynCode is installed from GitHub with the transformers version it requires.

In [1]:
!pip install -q --only-binary=:all: "transformers==4.53.2" "huggingface-hub>=0.30,<1.0" "datasets==3.6.0" tokenizers
!pip install -q --no-deps --ignore-requires-python git+https://github.com/structuredllm/syncode.git
!pip install -q interegular regex fire accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Grammar and Prompt

The grammar only allows `Operands: <float>, <float> Operator: <+ - * />`. The few-shot examples show the extraction only, with no final result.

In [2]:
import operator
import re

import torch
from syncode import Syncode

MODEL = "microsoft/phi-2"

GRAMMAR = r"""
start: "Operands:" " " NUM "," " " NUM " " "Operator:" " " OP

NUM: /[0-9]{1,9}\.[0-9]{1,9}/
OP: "+" | "-" | "*" | "/"
"""

FEW_SHOT_PROMPT = """Extract the two floating point operands and the operator from each query.

Query: What is 12.5 plus 7.25?
Operands: 12.5, 7.25 Operator: +

Query: What is 9. minus 2.5?
Operands: 9.0, 2.5 Operator: -

Query: What is 6.0 multiplied by 0.5?
Operands: 6.0, 0.5 Operator: *

Query: What is 81. divided by 3.0?
Operands: 81.0, 3.0 Operator: /

"""

# Model

Phi-2 inside SynCode in `grammar_strict` mode with greedy decoding. The first run builds the grammar mask store, which takes a short while.

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

llm = Syncode(
    model=MODEL,
    grammar=GRAMMAR,
    mode="grammar_strict",
    parse_output_only=True,
    device=device,
    max_new_tokens=50,
    do_sample=False,
)

[2026-10-26 05:33:04,081-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-26 05:35:52,086-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_1614761569_50257.pkl exist: False
[2026-10-26 05:35:52,087-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_1614761569_50257.pkl.
[2026-10-26 05:35:52,596-syncode.mask_store.fsm_set] - 6 FSMs with 49 states initialized in 0.00 seconds
[2026-10-26 05:35:52,597-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-26 05:35:52,605-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 36 to 8


100%|██████████| 49/49 [00:09<00:00,  5.31it/s]

[2026-10-26 05:36:01,870-syncode.mask_store.mask_store] - Time taken to create mask store: 9.78 seconds


# run_syncode

`run_syncode` builds the prompt, lets SynCode generate the extraction, parses it, and computes the answer with Python.

In [4]:
OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}
EXTRACTION = re.compile(r"Operands:\s*(\d+\.\d+),\s*(\d+\.\d+)\s*Operator:\s*([+\-*/])")


def run_syncode(textual_query):
    prompt = FEW_SHOT_PROMPT + f"Query: {textual_query.strip()}\n"
    generated = llm.infer(prompt)[0]
    match = EXTRACTION.search(generated)
    operands = [float(match.group(1)), float(match.group(2))]
    op = match.group(3)
    try:
        answer = float(OPS[op](operands[0], operands[1]))
    except ZeroDivisionError:
        answer = float("inf")
    return operands, op, answer

# Tests

The three examples from the problem statement, plus more phrasings.

In [5]:
tests = [
    ("What is 327. multiplied by 11.0?", ([327.0, 11.0], "*", 3597.0)),
    ("What is 45.1 plus 23.54?", ([45.1, 23.54], "+", 45.1 + 23.54)),
    ("What is 120.4 divided by 4.0?", ([120.4, 4.0], "/", 30.1)),
    ("What is 18.5 minus 6.25?", ([18.5, 6.25], "-", 12.25)),
    ("What is 7.5 times 4.0?", ([7.5, 4.0], "*", 30.0)),
    ("What is 100. divided by 8.0?", ([100.0, 8.0], "/", 12.5)),
]

passed = 0
for query, expected in tests:
    result = run_syncode(query)
    ok = result[0] == expected[0] and result[1] == expected[1] and abs(result[2] - expected[2]) < 1e-9
    passed += ok
    print("PASS" if ok else "FAIL", "|", query, "->", result)
print(f"{passed}/{len(tests)} passed")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


PASS | What is 327. multiplied by 11.0? -> ([327.0, 11.0], '*', 3597.0)
PASS | What is 45.1 plus 23.54? -> ([45.1, 23.54], '+', 68.64)
PASS | What is 120.4 divided by 4.0? -> ([120.4, 4.0], '/', 30.1)
PASS | What is 18.5 minus 6.25? -> ([18.5, 6.25], '-', 12.25)
PASS | What is 7.5 times 4.0? -> ([7.5, 4.0], '*', 30.0)
PASS | What is 100. divided by 8.0? -> ([100.0, 8.0], '/', 12.5)
6/6 passed
